# Talk To Data - Educational Notebook

**Learn AI-powered natural language to SQL**

This notebook demonstrates how to build a system that converts plain English questions into SQL queries using large language models (LLMs). Each cell represents one key concept in the pipeline.

## What You'll Learn
1. Load and explore data
2. Generate structured data semantics using AI
3. Define business logic and terminology
4. Provide reference queries for few-shot learning
5. Build a query engine that generates SQL from natural language
6. Execute queries and display results

## Prerequisites
- Python 3.9+
- OpenAI API key (or Anthropic API key)
- Basic SQL knowledge

## Setup
If running in Google Colab:
1. Go to **Secrets** (key icon in left sidebar)
2. Add secret: `OPENAI_API_KEY` with your API key
3. Run all cells below

## Cell 1: Setup & Imports

First, we install dependencies and configure our environment.

In [ ]:
!pip install -q openai pandas pyyaml

import os, sqlite3
import pandas as pd
import yaml
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except ImportError:
    api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError("API key not found. Set OPENAI_API_KEY in Colab Secrets or environment.")

client = OpenAI(api_key=api_key)
print("✅ Setup complete!")

## Cell 2: Data Ingestion

Load sample datasets (customers, orders, products) into pandas DataFrames, then create an in-memory SQLite database.

In [ ]:
base_url = "https://raw.githubusercontent.com/dave-melillo/talk-to-data-notebook/main/data/"

try:
    df_customers = pd.read_csv('data/customers.csv')
    df_orders = pd.read_csv('data/orders.csv')
    df_products = pd.read_csv('data/products.csv')
except FileNotFoundError:
    df_customers = pd.read_csv(base_url + 'customers.csv')
    df_orders = pd.read_csv(base_url + 'orders.csv')
    df_products = pd.read_csv(base_url + 'products.csv')

conn = sqlite3.connect(':memory:')
for name, df in [('customers', df_customers), ('orders', df_orders), ('products', df_products)]:
    df.to_sql(name, conn, index=False, if_exists='replace')
    print(f"📊 {name.title()} Table:")
    display(df.head())

print(f"\n✅ Loaded {len(df_customers)} customers, {len(df_orders)} orders, {len(df_products)} products")

## Cell 3: Data Semantic Generation (AI)

Use an LLM to analyze the database schema and generate structured metadata about each table and column. This helps the query engine understand what the data represents.

In [ ]:
def generate_data_semantic(df_dict):
    schema_info = {t: {'columns': list(df.columns), 'dtypes': df.dtypes.astype(str).to_dict(), 
                        'sample_rows': df.head(3).to_dict('records')} for t, df in df_dict.items()}
    
    prompt = f"""Analyze this database schema and generate YAML.

Schema:
{yaml.dump(schema_info, default_flow_style=False)}

Format:
tables:
  table_name:
    description: "..."
    columns:
      col: {{type: "...", description: "...", primary_key: true/false}}

Return ONLY YAML."""
    
    response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}], temperature=0)
    yaml_text = response.choices[0].message.content.strip()
    for fence in ['```yaml', '```']:
        if yaml_text.startswith(fence):
            yaml_text = yaml_text.split(fence)[1].split('```')[0].strip()
    return yaml.safe_load(yaml_text)

data_semantic = generate_data_semantic({'customers': df_customers, 'orders': df_orders, 'products': df_products})
print("🧠 Data Semantic:\n" + yaml.dump(data_semantic, default_flow_style=False))
print("✅ Generated!")

## Cell 4: Business Semantic (User Input)

Define business-specific terminology, KPIs, and rules. This helps the AI understand domain-specific concepts like "churned customer" or "lifetime value."

In [ ]:
business_semantic = {
    'glossary': {'churned_customer': "status = 'churned'", 'active_customer': "status = 'active'",
                 'high_value_customer': "lifetime_value > 10000", 'recent_order': "order_date within last 30 days"},
    'kpis': {'total_revenue': 'SUM(orders.order_total)', 'avg_order_value': 'AVG(orders.order_total)',
             'customer_count': 'COUNT(DISTINCT customers.customer_id)', 'total_orders': 'COUNT(orders.order_id)'},
    'rules': ["Join orders to customers via customer_id", "Join orders to products via product_id",
              "Use created_at for signup date", "Use order_date for timing"]
}

print("📝 Business Semantic:\n" + yaml.dump(business_semantic, default_flow_style=False))
print("✅ Defined!")

## Cell 5: Reference Queries (Few-Shot Learning)

Provide example question-SQL pairs to guide the LLM. These examples help the AI learn your preferred SQL style and query patterns.

In [ ]:
reference_queries = [
    {'question': 'Who are my top 10 customers by lifetime value?',
     'sql': 'SELECT customer_id, name, lifetime_value FROM customers ORDER BY lifetime_value DESC LIMIT 10'},
    {'question': 'What is the total revenue for March 2024?',
     'sql': "SELECT SUM(order_total) as total_revenue FROM orders WHERE order_date >= '2024-03-01' AND order_date < '2024-04-01'"},
    {'question': 'Which products have low stock (less than 100 units)?',
     'sql': 'SELECT product_id, name, stock_quantity FROM products WHERE stock_quantity < 100 ORDER BY stock_quantity ASC'},
    {'question': 'How many active customers do we have?',
     'sql': "SELECT COUNT(*) as active_customers FROM customers WHERE status = 'active'"},
    {'question': 'What are the best-selling products by quantity?',
     'sql': 'SELECT p.product_id, p.name, SUM(o.quantity) as total_sold FROM orders o JOIN products p ON o.product_id = p.product_id GROUP BY p.product_id, p.name ORDER BY total_sold DESC LIMIT 10'}
]

print("📚 Reference Queries:")
for i, r in enumerate(reference_queries, 1):
    sql = r['sql'][:80] + '...' if len(r['sql']) > 80 else r['sql']
    print(f"{i}. {r['question']}\n   {sql}\n")
print(f"✅ {len(reference_queries)} queries loaded")

## Cell 6: Query Engine (AI SQL Generation)

Combine data semantic + business semantic + reference queries to generate SQL from natural language questions.

In [ ]:
def ask_question(question, show_prompt=False):
    refs = '\n'.join([f"Q: {r['question']}\nSQL: {r['sql']}" for r in reference_queries])
    context = f"""SQL expert. Convert natural language to SQL.

SCHEMA:
{yaml.dump(data_semantic, default_flow_style=False)}

BUSINESS RULES:
{yaml.dump(business_semantic, default_flow_style=False)}

EXAMPLES:
{refs}

RULES: Return ONLY SQL, no explanations. Use SQLite syntax.

Q: {question}
SQL:"""
    
    if show_prompt:
        print("🔍 Prompt:\n" + "="*80 + "\n" + context + "\n" + "="*80)
    
    response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": context}], temperature=0)
    sql = response.choices[0].message.content.strip()
    for fence in ['```sql', '```']:
        if sql.startswith(fence):
            sql = sql.split(fence)[1].split('```')[0].strip()
    return sql

test_q = "Who are my top 5 customers by total spending?"
generated_sql = ask_question(test_q)
print(f"❓ {test_q}\n\n🤖 SQL:\n{generated_sql}\n\n✅ Query engine ready!")

## Cell 7: Execute Query & Display Results

Execute the generated SQL and display results as a pandas DataFrame.

In [ ]:
def execute_query(sql):
    try:
        return pd.read_sql_query(sql, conn)
    except Exception as e:
        print(f"❌ SQL Error: {e}\nSQL:\n{sql}")
        return None

results = execute_query(generated_sql)
if results is not None:
    print("📊 Results:")
    display(results)
    print(f"\n✅ {len(results)} rows returned")

## Cell 8: Interactive Query Interface

Now you can ask your own questions! Try these examples:
- "What were the total sales in March 2024?"
- "Which customers haven't ordered in the last 30 days?"
- "What is the average profit margin by product category?"
- "Who are the churned customers?"
- "What are the top 10 best-selling products?"

In [ ]:
def query(question, show_sql=True, show_prompt=False):
    print(f"❓ {question}\n")
    sql = ask_question(question, show_prompt=show_prompt)
    if show_sql:
        print(f"🤖 SQL:\n{sql}\n")
    results = execute_query(sql)
    if results is not None:
        print("📊 Results:")
        display(results)
        print(f"\n✅ {len(results)} rows")
    return results

query("What are the top 10 best-selling products by revenue?")

## Try Your Own Questions!

Modify the cell below to ask your own questions:

In [ ]:
query("How many orders were placed by high-value customers?")

## Advanced: Debug Mode

See the full prompt sent to the LLM by setting `show_prompt=True`:

In [ ]:
query("What is the average order value by customer status?", show_prompt=True)

## What's Next?

**Experiment with:**
- Different datasets (upload your own CSV files)
- Modified business semantics (add new KPIs)
- Additional reference queries (teach new patterns)
- Different LLM models (Claude, Gemini, local models)

**Learn more:**
- [Production version: talk-to-data](https://github.com/dave-melillo/talk-to-data)
- [OpenAI API Documentation](https://platform.openai.com/docs)
- [SQLite Tutorial](https://www.sqlitetutorial.net/)

---

**Built with ❤️ for learning and teaching.**